# Fixed-beta pairs parameter tuning

Tune only the trading parameters of the six pairs selected in days 0–300. This notebook evaluates configurations on days 300–500; it does not use days 500–750.

In [ ]:
from itertools import product
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd

DISCOVERY_END = 300
TUNING_END = 500

repo_root = Path.cwd()
if not (repo_root / 'backtester').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'backtester'))

from backtest.data import load_prices
from backtest.engine import run_backtest
from backtest.metrics import compute_metrics

prices_full = load_prices(str(repo_root / 'backtester' / 'data' / '2026' / 'prices.txt'))
prices = prices_full[:, :TUNING_END]

strategy_path = repo_root / 'src' / 'strategies' / 'pairs_6_fixed_beta.py'
spec = importlib.util.spec_from_file_location('pairs_6_fixed_beta', strategy_path)
strategy_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_module)

## Parameter grid

Keep this grid deliberately small. The six pairs remain fixed; only lookback, entry threshold, exit threshold, and time stop are compared.

In [ ]:
parameter_grid = {
    'lookback_days': [120, 180, 250],
    'entry_z': [1.25, 1.50, 1.75],
    'exit_z': [0.25, 0.50],
    'max_holding_days': [20, 30],
}
np.prod([len(values) for values in parameter_grid.values()])

## Validation on days 300–500

At the first validation call, each configuration calibrates only from the history available through day 300. No day after 500 is loaded or evaluated here.

In [ ]:
results = []
for lookback_days, entry_z, exit_z, max_holding_days in product(
    parameter_grid['lookback_days'],
    parameter_grid['entry_z'],
    parameter_grid['exit_z'],
    parameter_grid['max_holding_days'],
):
    strategy_module.LOOKBACK_DAYS = lookback_days
    strategy_module.ENTRY_Z = entry_z
    strategy_module.EXIT_Z = exit_z
    strategy_module.MAX_HOLDING_DAYS = max_holding_days
    strategy_module.reset_state()

    result = run_backtest(
        prices,
        strategy_module.getMyPosition,
        eval_start=DISCOVERY_END,
        eval_end=TUNING_END,
    )
    metrics = compute_metrics(result)
    results.append({
        'lookback_days': lookback_days,
        'entry_z': entry_z,
        'exit_z': exit_z,
        'max_holding_days': max_holding_days,
        'score': metrics['score'],
        'mean_pl': metrics['mean_pl'],
        'ann_sharpe': metrics['ann_sharpe'],
        'max_drawdown': metrics['max_drawdown'],
        'total_dvolume': metrics['total_dvolume'],
    })

tuning_results = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
display(tuning_results.head(15).style.format({
    'score': '{:.2f}', 'mean_pl': '{:.2f}', 'ann_sharpe': '{:.2f}',
    'max_drawdown': '{:.2f}', 'total_dvolume': '{:,.0f}',
}))

## Choose a robust configuration

Prefer a configuration that remains competitive across nearby settings, not merely the single highest row. Record one final choice, then freeze both pairs and parameters before any 500–750 evaluation.

In [ ]:
best_configuration = tuning_results.iloc[0]
best_configuration